# arange-fancy-index-cross-entropy — faded example 2: Complete the mean cross-entropy decomposition

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `arange-fancy-index-cross-entropy`. Running the beacon reports progress on the `Loss: arange fancy-index cross-entropy` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: arange fancy-index cross-entropy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arange-fancy-index-cross-entropy`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arange-fancy-index-cross-entropy"
DD_SUBTOPIC = "Loss: arange fancy-index cross-entropy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Mean cross-entropy is `(logsumexp(logits) - logits[arange(B), target]).mean()`. The logsumexp and reduction are given; you supply the `arange` fancy-index gather of the target logits.

## Faded exercise 2

Implement `ce_mean(logits, target)` so it equals `F.cross_entropy(logits, target, reduction='mean')`. The logsumexp term, the subtraction, and the mean are filled in. Complete only the line that gathers the per-sample target logit using the `arange` idiom. Do not call `F.cross_entropy` or `F.nll_loss`.

**Fill in:** Gather the `(B,)` vector of logits at the true class via `logits` indexed by `t.arange(B)` and `target`.

In [ ]:
def ce_mean(logits, target):
    B = logits.shape[0]
    lse = t.logsumexp(logits, dim=-1)             # (B,)
    picked = logits[t.arange(B), target]          # (B,)
    return (lse - picked).mean()                  # scalar

t.manual_seed(0)
logits = t.randn(5, 4)
target = t.tensor([1, 3, 0, 2, 1])
print(float(ce_mean(logits, target)))


def _test():
    t.manual_seed(2)
    logits = t.randn(9, 6)
    target = t.randint(0, 6, (9,))
    out = ce_mean(logits, target)
    assert out.shape == (), out.shape
    ref = t.nn.functional.cross_entropy(logits, target, reduction='mean')
    assert t.allclose(out, ref, atol=1e-5), (out, ref)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def ce_mean(logits, target):
    B = logits.shape[0]
    lse = t.logsumexp(logits, dim=-1)             # (B,)
    picked = logits[t.arange(B), target]          # (B,)
    return (lse - picked).mean()                  # scalar

t.manual_seed(0)
logits = t.randn(5, 4)
target = t.tensor([1, 3, 0, 2, 1])
print(float(ce_mean(logits, target)))
```
</details>